# Exp 03: Dynamic Shape Problem

目标：
1. 故意制造 `torch.compile` recompile，观察 `unique_graphs` 如何增长。
2. 理解 batch size 变化、`0/1 specialization`、Python scalar 参数为什么会触发重新编译。
3. 学会用 `TORCH_LOGS=recompiles` 读失败 guard。
4. 为下一节的 `mark_dynamic`、`drop_last`、bucketing 修复方案做铺垫。

推荐在脚本版中配合日志运行：

```bash
TORCH_LOGS=recompiles python experiments_py/exp03_dynamic_shape_problem.py
TORCH_LOGS=recompiles_verbose,dynamic python experiments_py/exp03_dynamic_shape_problem.py
```

notebook 中主要看 `counters["stats"]["unique_graphs"]`。

## 准备工作

`torch.compile` 会为输入 shape、Python 常量、模块属性等生成 guard。后续调用如果 guard 失败，Dynamo 会重新 trace/compile，产生新的 graph。

本实验刻意写出三类常见问题：

- batch size 变化且 `dynamic=False`：每种 shape 都可能独立编译。
- batch size 为 0 或 1：PyTorch 2.x 会特别处理这些维度，容易触发额外 specialization。
- Python `int` / `float` 作为 compiled function 参数：不同值会生成不同 `EQUALS_MATCH` guard。

In [ ]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch._dynamo.utils import compile_times, counters

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."

DEVICE = "cuda"
DTYPE = torch.bfloat16
IN_DIM = 512
OUT_DIM = 128
STEPS = 30

torch.manual_seed(42)

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(IN_DIM, 1024)
        self.fc2 = nn.Linear(1024, OUT_DIM)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.gelu(self.fc1(x)))


def clear_grads(model: nn.Module) -> None:
    for param in model.parameters():
        param.grad = None


def run_step(model: nn.Module, batch_size: int) -> None:
    x = torch.randn(batch_size, IN_DIM, device=DEVICE, dtype=DTYPE)
    y = torch.randn(batch_size, OUT_DIM, device=DEVICE, dtype=DTYPE)
    loss = F.mse_loss(model(x), y)
    loss.backward()
    clear_grads(model)


def print_compile_summary() -> None:
    print(f"unique_graphs = {counters['stats']['unique_graphs']}")
    try:
        text = compile_times(output="str")
        if text.strip():
            print(text)
    except Exception as exc:
        print(f"compile_times unavailable: {type(exc).__name__}: {exc}")

## 场景 A：变化 batch size + `dynamic=False`

`dynamic=False` 表示不要自动把变化维度升级成 symbolic shape。这样做能让问题更明显：batch 维每出现一个新值，都可能触发一次新的 graph。

真实训练中常见来源是 DataLoader 没有 `drop_last=True`，或数据预处理导致 batch / sequence length 抖动。

In [ ]:
model_a = SimpleNet().to(DEVICE, dtype=DTYPE)
compiled_a = torch.compile(model_a, mode="default", fullgraph=False, dynamic=False)

batch_sizes = [4, 8, 12, 16, 20, 24, 28, 32]
counters.clear()

for step, batch_size in enumerate((batch_sizes * 4)[:STEPS]):
    run_step(compiled_a, batch_size)
    if step < 10 or step % 5 == 0:
        print(
            f"step={step:2d}  batch={batch_size:2d}  "
            f"unique_graphs={counters['stats']['unique_graphs']}"
        )

torch.cuda.synchronize()
print("\n结果：不同 batch size 会不断增加 unique_graphs。")
print_compile_summary()

## 场景 B：`0/1 specialization`

PyTorch 编译器会特别处理 size 为 0 或 1 的维度，因为它们和广播、空 tensor 语义相关。即使 automatic dynamic 已经把普通 batch 维升级为 symbolic，`batch=1` 也经常需要单独 graph。

工程上常见建议是：训练 DataLoader 使用 `drop_last=True`，或者保证动态 batch 的 `min >= 2`。

In [ ]:
model_b = SimpleNet().to(DEVICE, dtype=DTYPE)
compiled_b = torch.compile(model_b, mode="default", fullgraph=False, dynamic=None)

counters.clear()
for step, batch_size in enumerate([8, 16, 24, 1, 32]):
    run_step(compiled_b, batch_size)
    print(
        f"step={step}  batch={batch_size:2d}  "
        f"unique_graphs={counters['stats']['unique_graphs']}"
    )

torch.cuda.synchronize()
print("\n观察 batch=1 是否带来额外 graph。")

## 场景 C：Python scalar 参数变化

Python `int` / `float` 不是 tensor。Dynamo 通常会把它们当作编译期常量，并生成类似 `scale == 2` 的 guard。

如果这个值每步变化，例如 learning-rate scheduler、temperature、top-k、某个 routing 参数，就可能导致频繁 recompile。修复方式通常是把它改成 0-dim tensor，或者把逻辑移到 compiled graph 外部。

In [ ]:
@torch.compile(fullgraph=False)
def fn_with_python_scale(x: torch.Tensor, scale: int) -> torch.Tensor:
    return x * scale + 1.0


x = torch.randn(32, IN_DIM, device=DEVICE, dtype=DTYPE)
counters.clear()

for scale in [1, 2, 3, 4, 5]:
    fn_with_python_scale(x, scale)
    print(f"scale={scale}  unique_graphs={counters['stats']['unique_graphs']}")

torch.cuda.synchronize()
print("\n每个不同 Python int 值都可能对应一个独立 guard。")

## 小结

三类 recompile 触发源：

1. `dynamic=False` + shape 变化：每种 shape 都可能生成新 graph。
2. `batch=0/1`：触发特殊 specialization，容易破坏 dynamic graph 复用。
3. Python scalar 参数变化：不同值触发 `EQUALS_MATCH` guard。

下一节会对应给出修复方案：`mark_dynamic`、automatic dynamic、unbacked symbol、`drop_last=True` 和 bucketing。